In [1]:
from pprint import pprint
import datasets as datasets_lib
import grain
import pandas as pd
import os
import fsspec
import numpy as np

import transformers
from tunix.generate import mappings

Dataset = datasets_lib.Dataset
AutoTokenizer = transformers.AutoTokenizer

from absl import logging as absl_logging

import logging
import sys

# ====== Logging Configuration ======
# 1. Force absl to use python logging
absl_logging.use_python_logging()

# 2. Configure the root logger
logging.basicConfig(
    stream=sys.stdout,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(name)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)

# 3. Explicitly set levels for relevant loggers
logging.getLogger().setLevel(logging.INFO)
logging.getLogger("absl").setLevel(logging.INFO)

# 4. Set absl verbosity
absl_logging.set_verbosity(absl_logging.INFO)
absl_logging.set_stderrthreshold("info")

print("Logging configured at INFO level.")
import os

os.environ["JAX_COMPILER_CACHE_TGTS"] = ""

/mnt/disks/linchai-data/miniconda3/envs/tunix/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Logging configured at INFO level.


In [2]:
try:
  from GOOGLE_INTERNAL_PACKAGE_PATH.pyglib import gfile
  from etils import ecolab

  cm = ecolab.adhoc(
      source=ecolab.FROM_NOTEBOOK_OR_HEAD,
      reload="tunix",
      behavior="preferred",
      cell_autoreload=True,
  )

  file_open = gfile.Open

  NOTEBOOK_ENV = "g3"
except Exception:
  NOTEBOOK_ENV = "git"

  import contextlib
  cm = contextlib.nullcontext()

  file_open = fsspec.open

In [3]:
with cm:
  from tunix.models.gemma4 import model as gemma4_lib
  from tunix.models.gemma4 import params_safetensors as gemma4_params_lib
  from tunix.generate import sampler as sampler_lib
  from tunix.utils import math_utils
# %%
from typing import Any, Dict, Optional
import jax
from jax import numpy as jnp
from flax import nnx
import orbax.checkpoint as ocp
from tqdm.auto import tqdm
import re


In [4]:

MODEL_PATH_PREFIX = "gs://tunix/models"
MODEL_MAPPING = {
    "google/gemma-4-26B-A4B-it": (
      gemma4_lib.ModelConfig.gemma4_26b_a4b(),
      "/mnt/disks/linchai-data/huggingface/hub/models--google--gemma-4-26B-A4B-it/snapshots/6e6f6edea8c52db2094dca3086e4b963a0034dfc",
    ),
    
}

In [5]:
import os
os.environ["TPU_LOG_DIR"] = "/mnt/disks/linchai-data/my_tpu_logs"
os.environ["SKIP_JAX_PRECOMPILE"] = "True"

In [6]:
model_version = "google/gemma-4-26B-A4B-it"
model_config, model_path = MODEL_MAPPING[model_version]
model_config.dtype = jnp.bfloat16
model_config.param_dtype = jnp.bfloat16

tokenizer = AutoTokenizer.from_pretrained(model_version)

2026-05-29 18:03:38 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/google/gemma-4-26B-A4B-it/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 18:03:38 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/gemma-4-26B-A4B-it/6e6f6edea8c52db2094dca3086e4b963a0034dfc/config.json "HTTP/1.1 200 OK"


2026-05-29 18:03:38 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/google/gemma-4-26B-A4B-it/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 18:03:38 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/gemma-4-26B-A4B-it/6e6f6edea8c52db2094dca3086e4b963a0034dfc/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-29 18:03:38 - INFO - [httpx] HTTP Request: GET https://huggingface.co/api/models/google/gemma-4-26B-A4B-it/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-29 18:03:38 - INFO - [httpx] HTTP Request: GET https://huggingface.co/api/models/google/gemma-4-26B-A4B-it/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-05-29 18:03:41 - INFO - [httpx] HTTP Request: GET https://huggingface.co/api/models/google/gemma-4-26B-A4B-it "HTTP/1.1 200 OK"


In [7]:

trainer_mesh = jax.sharding.Mesh(np.array(jax.devices()).reshape(1, 4), axis_names=["fsdp", "tp"])
rollout_mesh = jax.sharding.Mesh(np.array(jax.devices()).reshape(1, 4), axis_names=["fsdp", "tp"])


In [8]:

print("Loading model from safe tensors...")
with trainer_mesh:
  trainer_model = gemma4_params_lib.create_model_from_safe_tensors(file_dir=model_path, config=model_config, mesh=trainer_mesh)
  # ref_model = gemma4_params_lib.create_model_from_safe_tensors(file_dir=model_path, config=model_config, mesh=trainer_mesh)
  ref_model = trainer_model


Loading model from safe tensors...
2026-05-29 18:03:53 - INFO - [root] num_kv_heads=2 is not divisible by TP size 4, sharding k/v projections on head_dim instead of kv-heads.
2026-05-29 18:03:53 - INFO - [root] num_kv_heads=2 is not divisible by TP size 4, sharding k/v projections on head_dim instead of kv-heads.
2026-05-29 18:03:53 - INFO - [root] num_kv_heads=2 is not divisible by TP size 4, sharding k/v projections on head_dim instead of kv-heads.
2026-05-29 18:03:54 - INFO - [root] num_kv_heads=2 is not divisible by TP size 4, sharding k/v projections on head_dim instead of kv-heads.
2026-05-29 18:03:55 - INFO - [root] num_kv_heads=2 is not divisible by TP size 4, sharding k/v projections on head_dim instead of kv-heads.
2026-05-29 18:03:55 - WARNING - [absl] Skipped loading 356 keys because they could not be mapped to model weights. This may be expected, for example when loading only text weights from a multimodal checkpoint. Missing keys: [model.embed_vision.embedding_projection.

In [9]:

import optax
from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.rollout import base_rollout

import os
os.environ["NEW_MODEL_DESIGN"] = "true"
optimizer = optax.adamw(learning_rate=1e-6,)

base_rollout_dict = {
    "max_prompt_length": 32,
    "kv_cache_size": 32 + 16 + 256,
    "temperature": 1.0,
    "top_p": 1.0,
    "top_k": 0,
    "return_logprobs": True,
    "max_tokens_to_generate": 16,
}

vllm_rollout_dict = {
    # vllm-tpu specific configs
    "rollout_vllm_model_version": model_version,
    "rollout_vllm_hbm_utilization": 0.33,
    "rollout_vllm_tpu_backend_type": "jax",
    "rollout_vllm_server_mode": True,
    "rollout_vllm_async_scheduling": True,
    "rollout_vllm_init_with_random_weights": False,
    "rollout_vllm_max_num_seqs": 1,
    "rollout_vllm_max_num_batched_tokens": 2496,
    "rollout_vllm_kwargs": {
        "kv_cache_metrics": True,
        "disable_log_stats": False,
        "enable_prefix_caching": False,
        "dtype": "bfloat16",
    },
}
rollout_engine_config = base_rollout.RolloutConfig(
    **base_rollout_dict, **vllm_rollout_dict
)
cluster_config = rl_cluster_lib.ClusterConfig(
    role_to_mesh={
        rl_cluster_lib.Role.ACTOR: trainer_mesh,
        rl_cluster_lib.Role.REFERENCE: trainer_mesh,
        rl_cluster_lib.Role.ROLLOUT: rollout_mesh,
    },
    rollout_engine="vllm",
    offload_to_cpu=False,
    training_config=rl_cluster_lib.RLTrainingConfig(
        actor_optimizer=optimizer,
        eval_every_n_steps=5,
    ),
    rollout_config=rollout_engine_config,
)

tokenizer = AutoTokenizer.from_pretrained(model_path)

rl_cluster = rl_cluster_lib.RLCluster(
    actor=trainer_model,
    reference=ref_model,
    tokenizer=tokenizer,
    cluster_config=cluster_config,
)

INFO 05-29 18:04:11 [__init__.py:59] TPU info: node_name=maxtext-single-host-1-v5p-8 | tpu_type=v5p-8 | worker_id=0 | num_chips=4 | num_cores_per_chip=2
2026-05-29 18:04:11 - WARNING - [absl] Tensorflow library not found, tensorflow.io.gfile operations will use native shim calls. GCS paths (i.e. 'gs://...') cannot be accessed.


/mnt/disks/linchai-data/miniconda3/envs/tunix/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/mnt/disks/linchai-data/miniconda3/envs/tunix/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


INFO 05-29 18:04:16 [importing.py:46] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 05-29 18:04:16 [importing.py:81] Triton not installed or not compatible; certain GPU-related functions will not be available.


2026-05-29 18:04:17,719	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-05-29 18:04:18 - INFO - [absl] Engine kwargs setting key 'model' with value 'google/gemma-4-26B-A4B-it'.
2026-05-29 18:04:18 - INFO - [absl] Engine kwargs setting key 'max_model_len' with value '304'.
2026-05-29 18:04:18 - INFO - [absl] Engine kwargs setting key 'async_scheduling' with value 'True'.
2026-05-29 18:04:18 - INFO - [absl] Engine kwargs setting key 'max_num_batched_tokens' with value '2496'.
2026-05-29 18:04:18 - INFO - [absl] Engine kwargs setting key 'max_num_seqs' with value '1'.
2026-05-29 18:04:18 - INFO - [absl] Engine kwargs setting key 'hf_config_path' with value 'None'.
2026-05-29 18:04:18 - INFO - [absl] Engine kwargs setting key 'max_logprobs' with value '1'.
2026-05-29 18:04:18 - INFO - [absl] Engine kwargs setting key 'logprobs_mode' with value 'processed_logprobs'.
2026-05-29 18:04:18 - INFO - [absl] Engine kwargs setting key 'kv_cache_metrics' with value 'True'.
2026-05-29 18:04:18 - INFO - [absl] Engine kwargs setting key 'disable_log_stats' with value 

...

E0529 18:09:04.237578 1956713 cpu_aot_loader.cc:195] Loading XLA:CPU AOT result. Target machine feature +prefer-no-gather is not  supported on the host machine. Machine type used for XLA:CPU compilation doesn't match the machine type for execution. Compile machine features: [+64bit,+adx,+aes,+amx-bf16,+amx-int8,+amx-tile,+avx,+avx2,+avx512bf16,+avx512bitalg,+avx512bw,+avx512cd,+avx512dq,+avx512f,+avx512fp16,+avx512ifma,+avx512vbmi,+avx512vbmi2,+avx512vl,+avx512vnni,+avx512vpopcntdq,+avxvnni,+bmi,+bmi2,+cldemote,+clflushopt,+clwb,+cmov,+crc32,+cx16,+cx8,+f16c,+fma,+fsgsbase,+fxsr,+gfni,+invpcid,+lzcnt,+mmx,+movbe,+movdir64b,+movdiri,+pclmul,+popcnt,+prefer-no-gather,+prefer-no-scatter,+prfchw,+rdpid,+rdrnd,+rdseed,+rtm,+sahf,+serialize,+sha,+sse,+sse2,+sse3,+sse4.1,+sse4.2,+ssse3,+tsxldtrk,+vaes,+vpclmulqdq,+xsave,+xsavec,+xsaveopt,+xsaves,-amx-avx512,-amx-complex,-amx-fp16,-amx-fp8,-amx-movrs,-amx-tf32,-avx10.1,-avx10.2,-avx512vp2intersect,-avxifma,-avxneconvert,-avxvnniint16,-avxvnnii

INFO 05-29 18:09:05 [moe_weights.py:249] w13_weight_w1 shape after padding: (128, 2816, 1024)
INFO 05-29 18:09:05 [moe_weights.py:250] w13_weight_w3 shape after padding: (128, 2816, 1024)


E0529 18:09:07.197816 1956713 cpu_aot_loader.cc:195] Loading XLA:CPU AOT result. Target machine feature +prefer-no-gather is not  supported on the host machine. Machine type used for XLA:CPU compilation doesn't match the machine type for execution. Compile machine features: [+64bit,+adx,+aes,+amx-bf16,+amx-int8,+amx-tile,+avx,+avx2,+avx512bf16,+avx512bitalg,+avx512bw,+avx512cd,+avx512dq,+avx512f,+avx512fp16,+avx512ifma,+avx512vbmi,+avx512vbmi2,+avx512vl,+avx512vnni,+avx512vpopcntdq,+avxvnni,+bmi,+bmi2,+cldemote,+clflushopt,+clwb,+cmov,+crc32,+cx16,+cx8,+f16c,+fma,+fsgsbase,+fxsr,+gfni,+invpcid,+lzcnt,+mmx,+movbe,+movdir64b,+movdiri,+pclmul,+popcnt,+prefer-no-gather,+prefer-no-scatter,+prfchw,+rdpid,+rdrnd,+rdseed,+rtm,+sahf,+serialize,+sha,+sse,+sse2,+sse3,+sse4.1,+sse4.2,+ssse3,+tsxldtrk,+vaes,+vpclmulqdq,+xsave,+xsavec,+xsaveopt,+xsaves,-amx-avx512,-amx-complex,-amx-fp16,-amx-fp8,-amx-movrs,-amx-tf32,-avx10.1,-avx10.2,-avx512vp2intersect,-avxifma,-avxneconvert,-avxvnniint16,-avxvnnii

INFO 05-29 18:09:08 [default_loader.py:397] Loading weights took 192.86 seconds


INFO 05-29 18:09:09 [tpu_runner.py:725] Cleared JIT caches after weight loading
INFO 05-29 18:09:09 [tpu_runner.py:727] Init model | hbm=[(27.26, 95.74), (27.26, 95.74), (27.26, 95.74), (27.26, 95.74)]GiB
INFO 05-29 18:09:09 [tpu_platform.py:305] Using cache_config.block_size: 32 instead of overriding with _align_hybrid_block_size() since we set mamba_page_size_padded in kv_cache_manager.py
INFO 05-29 18:09:09 [tpu_worker.py:424] Memory statistics | total_hbm_limit_gb=382.97GiB | total_hbm_limit_cap_gb=126.38GiB | total_hbm_used_gb=109.02GiB | total_hbm_avail_gb=17.36GiB
INFO 05-29 18:09:09 [kv_cache_utils.py:1733] GPU KV cache size: 72,048 tokens
INFO 05-29 18:09:09 [kv_cache_utils.py:1734] Maximum concurrency for 304 tokens per request: 237.00x
INFO 05-29 18:09:11 [kv_cache_manager.py:875] Hybrid KV cache layout: num_kv_cache_groups=1, num_kv_cache_tensors=30, kv_cache_config.num_blocks=2370, duplicate_shared_layers=False
INFO 05-29 18:09:11 [kv_cache_manager.py:903] Init kv-cache | 

In [10]:
prompts = [{"role": "user", "content": "which is bigger, 1 or 2"}]
prompts = tokenizer.apply_chat_template(prompts, add_generation_prompt=True, tokenize=False)
out_data = rl_cluster.generate(prompts=prompts, max_generation_steps=16)
print("Text from VLLM sampler: ", out_data.text)
print("logprobs from VLLM sampler: ", out_data.logprobs)
print("tokens from VLLM sampler: ", out_data.tokens)
print("logits from VLLM sampler: ", out_data.logits)
print("left_padded_prompt_tokens from VLLM sampler: ", out_data.left_padded_prompt_tokens)

WARNING 05-29 18:09:21 [model.py:1509] Default vLLM sampling parameters have been overridden by the model's `generation_config.json`: `{'temperature': 1.0, 'top_k': 64, 'top_p': 0.95}`. If this is not intended, please relaunch vLLM instance with `--generation-config vllm`.
WARNING 05-29 18:09:21 [input_processor.py:274] Passing raw prompts to InputProcessor is deprecated and will be removed in v0.18. You should instead pass the outputs of Renderer.render_cmpl() or Renderer.render_chat().
Text from VLLM sampler:  ['2 is bigger than 1.<turn|>']
logprobs from VLLM sampler:  [[-0.34844642877578735, -1.6689251651769155e-06, 0.0, -4.267592521500774e-05, 0.0, 0.0, -1.1920901954454166e-07, -7.152541456889594e-07]]
tokens from VLLM sampler:  [array([236778,    563,  12869,   1082, 236743, 236770, 236761,    106],
      dtype=int32)]
logits from VLLM sampler:  None
left_padded_prompt_tokens from VLLM sampler:  [[     0      0      0      0      0      0      0      0      0      0
       2    10

In [11]:
out_tokens = out_data.tokens[0]


prompt_tokens = out_data.left_padded_prompt_tokens[0]
completion_tokens = out_tokens
print(f"{prompt_tokens=}")
print(f"{completion_tokens=}")
prompt_tokens = jnp.repeat(jnp.array([prompt_tokens]), 4, axis=0)
completion_tokens = jnp.repeat(jnp.array([completion_tokens]), 4, axis=0)

from tunix.rl import common
graphdef, state = nnx.split(trainer_model)

batch_size = prompt_tokens.shape[0]
if batch_size == 0:
  raise ValueError(
      "Cannot get reference log probabilities from an empty batch."
  )

from tunix.sft import sharding_utils
with jax.set_mesh(trainer_mesh):
    # This assumes reference model shards same data sharding as actor, which
    # should be true as ref model and policy model shares same architecture.
    dest_prompt_tokens = sharding_utils.shard_input(
        prompt_tokens,
         ("fsdp",),
    )
    dest_completion_tokens = sharding_utils.shard_input(
        completion_tokens,
         ("fsdp",),
    )
    dest_completion_mask = None

ref_logps_logits = common.compute_per_token_logps(
    graphdef,
    state,
    prompt_tokens=dest_prompt_tokens,
    completion_tokens=dest_completion_tokens,
    pad_id=rl_cluster.tokenizer.pad_id(),
    eos_id=rl_cluster.tokenizer.eos_id(),
    temperature = 1.0,
    return_logits = True,
)

prompt_tokens=array([     0,      0,      0,      0,      0,      0,      0,      0,
            0,      0,      2,    105,   2364,    107,   7650,    563,
        12869, 236764, 236743, 236770,    653, 236743, 236778,    106,
          107,    105,   4368,    107,    100,  45518,    107,    101],
      dtype=int32)
completion_tokens=array([236778,    563,  12869,   1082, 236743, 236770, 236761,    106],
      dtype=int32)
INFO 05-29 18:09:32 [loggers.py:271] Engine 000: Avg prompt throughput: 2.2 tokens/s, Avg generation throughput: 0.8 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
INFO 05-29 18:09:42 [loggers.py:271] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


In [12]:
import jax
jax.clear_caches()
jax.block_until_ready(ref_logps_logits)
print(f"Logits {ref_logps_logits[1] = }")
print(f"Logprobs {ref_logps_logits[0] = }")


Logits ref_logps_logits[1] = Array([[[-13.616589  ,  -7.11207   ,  -6.011297  , ..., -13.666176  ,
         -13.7650385 , -14.059365  ],
        [-12.759508  ,  -7.318154  ,  -9.868935  , ..., -12.963564  ,
         -12.861723  , -12.861723  ],
        [-20.99031   , -16.022356  , -17.150091  , ..., -20.926311  ,
         -21.053928  , -21.180017  ],
        ...,
        [-15.479143  ,  -4.0071487 ,  -9.138024  , ..., -15.1091175 ,
         -15.387218  , -15.387218  ],
        [-13.014357  ,  -7.5820704 ,  -9.813159  , ..., -13.216583  ,
         -13.115653  , -12.912691  ],
        [-11.398453  ,  -0.21191102,  -5.1364155 , ..., -11.022052  ,
         -11.237633  , -11.451896  ]],

       [[-13.5669155 ,  -7.082571  ,  -6.2807407 , ..., -13.666176  ,
         -13.7650385 , -14.059365  ],
        [-12.656958  ,  -7.523542  , -10.312373  , ..., -12.861723  ,
         -12.759508  , -12.759508  ],
        [-20.926311  , -16.022356  , -17.234049  , ..., -20.86198   ,
         -20.99031   ,

In [ ]:
logprobs_array = jnp.array(out_data.logprobs)
diff_mean = jnp.mean(jnp.abs(ref_logps_logits[0] - logprobs_array))
diff_std = jnp.std(jnp.abs(ref_logps_logits[0] - logprobs_array))
print(f"Mean absolute difference between reference log probabilities and VLLM sampler log probabilities: {diff_mean}")
print(f"Standard deviation of absolute difference: {diff_std}")

Mean absolute difference between reference log probabilities and VLLM sampler log probabilities: 0.005246538668870926
Standard deviation of absolute difference: 0.016025658696889877


: 